In [1]:
import requests
import pandas as pd
import time

In [2]:
API_KEY = "R1KFGHA9Z8R18038"
symbol = "AAPL"

url = f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&outputsize=compact&apikey={API_KEY}"

In [13]:
response = requests.get(url)
data = response.json()

# Extracting only the daily time series section from the JSON
time_series = data["Time Series (Daily)"]

# Turning the JSON data into a pandas dataframe
# orient="index" keeps the dates as the dataframe index
df = pd.DataFrame.from_dict(time_series, orient="index")
df

,1. open,2. high,3. low,4. close,5. volume
2026-05-13,293.5000,300.9200,293.5000,298.8700,52684260
2026-05-12,292.5600,295.2700,292.5600,294.8000,45748129
2026-05-11,291.9790,293.8800,290.2300,292.6800,42247285
2026-05-08,290.0100,294.7600,290.0000,293.3200,52692761
2026-05-07,289.2700,292.1300,285.7800,287.4400,45224300
...,...,...,...,...,...
2025-12-24,272.3400,275.4300,272.1950,273.8100,17910574
2025-12-23,270.8400,272.5000,269.5600,272.3600,29641999
2025-12-22,272.8600,273.8800,270.5050,270.9700,36571827
2025-12-19,272.1450,274.6000,269.9000,273.6700,144632048


In [5]:
# Rename columns
df = df.rename(columns={
    "1. open": "open",
    "2. high": "high",
    "3. low": "low",
    "4. close": "close",
    "5. volume": "volume"
})

# Convert index to datetime
df.index = pd.to_datetime(df.index)

# Sort by date
df = df.sort_index()

# Convert columns to numeric
numeric_cols = ["open", "high", "low", "close", "volume"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Handle missing values
df = df.ffill()

# Remove duplicate dates
df = df[~df.index.duplicated(keep="first")]

# Convert index into proper date column
df.reset_index(inplace=True)

# Rename index column to date
df.rename(columns={"index": "date"}, inplace=True)

df.head()

,date,open,high,low,close,volume
0,2025-12-18,273.605,273.63,266.950,272.19,51630721
1,2025-12-19,272.145,274.60,269.900,273.67,144632048
2,2025-12-22,272.860,273.88,270.505,270.97,36571827
3,2025-12-23,270.840,272.50,269.560,272.36,29641999
4,2025-12-24,272.340,275.43,272.195,273.81,17910574


In [6]:
# Daily return
df["daily_return"] = df["close"].pct_change()

# 7-day moving average
df["rolling_avg_7"] = df["close"].rolling(window=7).mean()

# Volatility
df["volatility"] = df["daily_return"].rolling(window=7).std()

df.head()

,date,open,high,low,close,volume,daily_return,rolling_avg_7,volatility
0,2025-12-18,273.605,273.63,266.950,272.19,51630721,NaN,NaN,NaN
1,2025-12-19,272.145,274.60,269.900,273.67,144632048,0.005437,NaN,NaN
2,2025-12-22,272.860,273.88,270.505,270.97,36571827,-0.009866,NaN,NaN
3,2025-12-23,270.840,272.50,269.560,272.36,29641999,0.005130,NaN,NaN
4,2025-12-24,272.340,275.43,272.195,273.81,17910574,0.005324,NaN,NaN


In [7]:
portfolio = pd.DataFrame({
    "asset": ["AAPL", "MSFT", "GOOGL"],
    "allocation": [0.4, 0.35, 0.25]
})

portfolio

,asset,allocation
0,AAPL,0.40
1,MSFT,0.35
2,GOOGL,0.25


In [8]:
# Add asset column
df["asset"] = "AAPL"

# Merge with portfolio table
final_df = df.merge(portfolio, on="asset", how="left")

# Weighted return
final_df["weighted_return"] = (
    final_df["daily_return"] * final_df["allocation"]
)

final_df.head()

,date,open,high,low,close,volume,daily_return,rolling_avg_7,volatility,asset,allocation,weighted_return
0,2025-12-18,273.605,273.63,266.950,272.19,51630721,NaN,NaN,NaN,AAPL,0.4,NaN
1,2025-12-19,272.145,274.60,269.900,273.67,144632048,0.005437,NaN,NaN,AAPL,0.4,0.002175
2,2025-12-22,272.860,273.88,270.505,270.97,36571827,-0.009866,NaN,NaN,AAPL,0.4,-0.003946
3,2025-12-23,270.840,272.50,269.560,272.36,29641999,0.005130,NaN,NaN,AAPL,0.4,0.002052
4,2025-12-24,272.340,275.43,272.195,273.81,17910574,0.005324,NaN,NaN,AAPL,0.4,0.002130


In [14]:
# Converting calculated columns into proper numeric format
# This helps Power BI recognise them as numbers instead of text

calc_cols = ["daily_return", "rolling_avg_7", "volatility"]

for col in calc_cols:
    final_df[col] = pd.to_numeric(final_df[col], errors="coerce")

# Filling NaN values created by rolling calculations
# Example:
# - first daily return has no previous day to compare to
# - first 6 rolling averages don't have enough rows yet

final_df = final_df.fillna(0)

# Quick check of data types
print(final_df.dtypes)

date               datetime64[ns]
open                      float64
high                      float64
low                       float64
close                     float64
volume                      int64
daily_return              float64
rolling_avg_7             float64
volatility                float64
asset                      object
allocation                float64
weighted_return           float64
dtype: object


In [15]:
final_df.to_csv(
    r"C:\Users\Admin\Desktop\projects\Automated Financial Data Pipeline\stock_prices.csv",
    index=False
)

print("stock_prices.csv exported successfully")

stock_prices.csv exported successfully


In [16]:
summary = final_df.groupby("asset").agg({
    "close": "last",
    "daily_return": "mean",
    "volatility": "mean"
}).reset_index()

summary

,asset,close,daily_return,volatility
0,AAPL,298.87,0.001047,0.013358


In [17]:
summary.to_csv(
    r"C:\Users\Admin\Desktop\projects\Automated Financial Data Pipeline\portfolio_summary.csv",
    index=False
)

print("portfolio_summary.csv exported successfully")

portfolio_summary.csv exported successfully
